# 03 · Stage 2 — model training & Optuna tuning
Default task: Priority vs Best Performer among high-access districts, with a class-1 threshold tuned on out-of-fold predictions for recall ≥ 0.82.
The full tuning run (`python -m src.run_pipeline`) writes `models/benchmark_metrics.json`. This notebook reports that benchmark; set `RETRAIN = True` to re-run training here (50 trials × 4 models takes several minutes).

In [1]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [2]:
RETRAIN = False
if RETRAIN:
    from src.run_pipeline import run
    run(trials=config["modelling"]["optuna_trials"])
metrics = json.loads(P("metrics").read_text())
rows = []
for name, r in metrics["candidate_models"].items():
    h, o = r["holdout"], r["out_of_fold"]
    rows.append({"model": name, "treeshap": r["treeshap_compatible"], "trials": r["optuna_trials"], "cv_objective": r["cv_selection_score"],
                 "oof_c1_recall": o["class_1_recall"], "holdout_c1_recall": h["class_1_recall"], "holdout_c1_f1": h["class_1_f1"],
                 "holdout_c1_f2": h["class_1_f2"], "holdout_macro_f1": h["macro_f1"], "holdout_roc_auc": h["ovr_roc_auc"],
                 "holdout_pr_auc": h["ovr_pr_auc"], "holdout_c1_brier": h.get("class_1_brier"),
                 "holdout_c1_recall@thr": r.get("holdout_at_threshold", {}).get("class_1_recall"),
                 "holdout_c1_precision@thr": r.get("holdout_at_threshold", {}).get("class_1_precision")})
print("Task:", metrics.get("task"), "| decision threshold:", metrics.get("decision_threshold"), "|", metrics.get("threshold_selection", {}).get("rule"))
print("Selected:", metrics["selected_model"], "|", metrics["selection_rule"])
pd.DataFrame(rows).round(3)

Task: discordance_binary | decision threshold: 0.3477 | highest threshold with OOF class-1 recall >= 0.82
Selected: random_forest | highest mean 5-fold CV objective (0.60 x class-1 F1 + 0.40 x macro F1) among TreeSHAP-compatible models; holdout is reported, never used for selection


,model,treeshap,trials,cv_objective,oof_c1_recall,holdout_c1_recall,holdout_c1_f1,holdout_c1_f2,holdout_macro_f1,holdout_roc_auc,holdout_pr_auc,holdout_c1_brier,holdout_c1_recall@thr,holdout_c1_precision@thr
0,random_forest,True,50,0.591,0.672,0.688,0.611,0.655,0.738,0.804,0.566,0.152,0.875,0.424
1,gradient_boosting,True,50,0.551,0.609,0.562,0.486,0.529,0.651,0.788,0.530,0.163,NaN,NaN
2,xgboost,True,50,0.558,0.750,0.750,0.533,0.645,0.656,0.782,0.626,0.196,NaN,NaN
3,lightgbm,True,50,0.549,0.641,0.812,0.634,0.730,0.741,0.803,0.612,0.174,NaN,NaN


In [3]:
selected = metrics["candidate_models"][metrics["selected_model"]]
cm = selected.get("holdout_at_threshold", selected["holdout"])["confusion_matrix"]
pd.DataFrame(cm["matrix"], index=[f"obs {l}" for l in cm["labels"]], columns=[f"pred {l}" for l in cm["labels"]])

,pred 1,pred 3
obs 1,14,2
obs 3,19,35


In [4]:
cal = metrics["calibration"]
print("Class-1 Brier raw: %.4f -> calibrated: %.4f" % (cal["holdout_class_1_brier_raw"], cal["holdout_class_1_brier_calibrated"]))
selected["best_params"]

Class-1 Brier raw: 0.1522 -> calibrated: 0.1327


{'n_estimators': 200,
 'max_depth': 13,
 'min_samples_leaf': 5,
 'max_features': 'log2'}

## Robustness: state-grouped CV and label sensitivity
Written by `python -m src.models.validate` (also run by the pipeline). Whole states are held out; thresholds are re-chosen inside each training fold.

In [5]:
validation = json.loads(P("validation_report").read_text())
pd.DataFrame({s: r["fold_mean"] for s, r in validation["cv"].items()}).T.round(3)

,class_1_recall,class_1_precision,class_1_f2,class_1_roc_auc
random_stratified,0.838,0.417,0.691,0.781
state_grouped,0.728,0.416,0.615,0.710


In [6]:
pd.DataFrame(validation["per_state_recall"]).round(2)

,state,n_priority,detected,recall_when_state_held_out
0,Sikkim,3,0,0.00
1,Telangana,4,0,0.00
2,Punjab,7,4,0.57
3,Assam,3,2,0.67
4,Chhattisgarh,4,3,0.75
5,Odisha,5,4,0.80
6,Rajasthan,10,8,0.80
7,Madhya Pradesh,10,9,0.90
8,Andaman & Nicobar Island,3,3,1.00
9,Gujarat,3,3,1.00


In [7]:
pd.DataFrame(validation["label_sensitivity"]).drop(columns="settings").round(3)

,variant,n_priority,priority_jaccard_vs_baseline,baseline_priority_retained,all_class_agreement
0,baseline,80,1.000,1.000,1.000
1,mortality_only,80,1.000,1.000,1.000
2,care_threshold_80pct,163,0.491,1.000,0.675
3,residual_threshold_0,117,0.684,1.000,0.947
4,residual_threshold_0.5,59,0.738,0.738,0.970
5,eb_prior_1000,79,0.916,0.950,0.986
6,eb_prior_10000,83,0.964,1.000,0.980
